# pyLSTemp - Exemplo de `spectral_index(...)`

Este notebook demonstra como calcular indices espectrais usando a funcao publica `spectral_index(...)`.

Neste exemplo sao calculados:

- `NDVI`: usa as bandas NIR e Red.
- `EVI`: usa as bandas NIR, Red e Blue.

Os arquivos `.tif` usados estao na pasta `data/` deste diretorio.

## 1. Instalacao

Execute esta celula se estiver em um ambiente novo, como Google Colab. Se a biblioteca ja estiver instalada, voce pode pular esta etapa.

In [ ]:
# Em Colab/Jupyter, descomente se precisar instalar.
# !pip install --quiet --upgrade git+https://github.com/daciocambraia/pyLSTemp.git rasterio

## 2. Importacoes

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pylstemp import spectral_index, list_algorithms

import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 3. Conferir indices disponiveis

In [ ]:
catalog = list_algorithms()

print("Familias disponiveis:")
print(list(catalog.keys()))

print("\nIndices espectrais:")
print(list(catalog["spectral_index"].keys()))

## 4. Localizar e ler as bandas opticas

Para Landsat 8/9, normalmente:

- `B2`: Blue.
- `B4`: Red.
- `B5`: NIR.

Use bandas ja alinhadas espacialmente e preferencialmente convertidas para reflectancia quando estiver fazendo analise fisica.

In [ ]:
data_dir = Path("data")

blue_path = data_dir / "LC82210712016107LGN01_B2.tif"
red_path = data_dir / "LC82210712016107LGN01_B4.tif"
nir_path = data_dir / "LC82210712016107LGN01_B5.tif"

for path in [blue_path, red_path, nir_path]:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo nao encontrado: {path}")

with rasterio.open(blue_path) as src:
    blue = src.read(1).astype(float)
    profile = src.profile.copy()

with rasterio.open(red_path) as src:
    red = src.read(1).astype(float)

with rasterio.open(nir_path) as src:
    nir = src.read(1).astype(float)

print("Blue shape:", blue.shape)
print("Red shape:", red.shape)
print("NIR shape:", nir.shape)

## 5. Criar mascara de pixels invalidos

Neste exemplo, valores `0` em qualquer banda sao tratados como pixels invalidos.

In [ ]:
mask = (blue == 0) | (red == 0) | (nir == 0)
print("Pixels invalidos:", int(mask.sum()))

## 6. Calcular NDVI e EVI

A chamada principal segue o padrao:

`spectral_index(index="ndvi", nir=nir, red=red, mask=mask)`

Para EVI, acrescente a banda `blue`.

In [ ]:
ndvi_image = spectral_index(index="ndvi", nir=nir, red=red, mask=mask)
evi_image = spectral_index(index="evi", nir=nir, red=red, blue=blue, mask=mask)

print("NDVI min/max/mean:", np.nanmin(ndvi_image), np.nanmax(ndvi_image), np.nanmean(ndvi_image))
print("EVI min/max/mean:", np.nanmin(evi_image), np.nanmax(evi_image), np.nanmean(evi_image))

## 7. Visualizar resultados

In [ ]:
def show_raster(image, title, cmap="RdYlGn"):
    plt.figure(figsize=(8, 6))
    plt.imshow(image, cmap=cmap, vmin=-1, vmax=1)
    plt.colorbar(label=title)
    plt.title(title)
    plt.axis("off")
    plt.show()

show_raster(ndvi_image, "NDVI")
show_raster(evi_image, "EVI")

## 8. Salvar resultado opcionalmente

A celula abaixo fica comentada para evitar escrita acidental.

In [ ]:
# output_path = Path("ndvi.tif")
# output_profile = profile.copy()
# output_profile.update(dtype="float32", nodata=np.nan, count=1)
#
# with rasterio.open(output_path, "w", **output_profile) as dst:
#     dst.write(ndvi_image.astype("float32"), 1)
#
# print("Arquivo salvo em:", output_path)